# 04 - PostgreSQL dan SQL Analytics (FR-05)
**Prasyarat:** PostgreSQL aktif (lokal), `.env` terisi (salin dari `.env.example`), database `bank_campaign_db` sudah dibuat. Catatan: notebook ini paling mudah dijalankan di VS Code/Jupyter lokal, karena Colab tidak bisa mengakses PostgreSQL di komputer Anda.

- [ ] Jalankan `sql/schema.sql`
- [ ] Impor data bersih ke tabel `bank_campaign`
- [ ] Verifikasi jumlah baris = data sumber
- [ ] Jalankan query di `sql/campaign_analysis.sql` dan tulis interpretasi

In [ ]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL          # perbaikan: import URL yang hilang
from sqlalchemy.exc import OperationalError

ROOT       = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN_PATH = ROOT / "data" / "processed" / "bank_campaign_clean.csv"
SQL_DIR    = ROOT / "sql"
ENV_PATH   = ROOT / ".env"



In [ ]:
load_dotenv(ENV_PATH, override=False)
keys = ["POSTGRES_USER", "POSTGRES_PASSWORD", "POSTGRES_HOST", "POSTGRES_PORT", "POSTGRES_DB"]
missing = [key for key in keys if not os.getenv(key)]
if missing:
    raise EnvironmentError(
        f"Environment variable belum diatur: {missing}. "
        "Buat file .env dari .env.example dan isi konfigurasi PostgreSQL."
    )

user, password, host, port, database = (os.getenv(key) for key in keys)
try:
    database_url = URL.create(
        "postgresql+psycopg",
        username=user,
        password=password,
        host=host,
        port=int(port),
        database=database,
    )
except (TypeError, ValueError) as error:
    raise ValueError("POSTGRES_PORT harus berupa nomor port yang valid.") from error

engine = create_engine(database_url, pool_pre_ping=True)
try:
    with engine.connect() as connection:
        print(connection.execute(text("SELECT version()")).scalar())
except OperationalError as error:
    raise ConnectionError(
        "Koneksi PostgreSQL gagal. Pastikan service aktif, database tersedia, "
        "dan konfigurasi host/port/kredensial pada .env benar."
    ) from error

In [ ]:
ddl = (SQL_DIR / "schema.sql").read_text(encoding="utf-8")
ddl_without_comments = "\n".join(
    line for line in ddl.splitlines() if not line.lstrip().startswith("--")
)
ddl_statements = [statement.strip() for statement in ddl_without_comments.split(";") if statement.strip()]
if not ddl_statements:
    raise ValueError("Tidak ada perintah DDL di sql/schema.sql")

with engine.begin() as connection:
    for statement in ddl_statements:
        connection.exec_driver_sql(statement)
print(f"Skema siap: {len(ddl_statements)} perintah DDL dijalankan.")

In [ ]:
df = pd.read_csv(CLEAN_PATH)
print("Baris di CSV:", len(df))

# Tabel ini khusus untuk dataset proyek; muat ulang secara transaksional agar notebook aman diulang.
# Batch 1.000 baris membatasi jumlah parameter per perintah INSERT.
with engine.begin() as connection:
    connection.execute(text("DELETE FROM bank_campaign"))
    df.to_sql(
        "bank_campaign", connection, if_exists="append", index=False,
        method="multi", chunksize=1000,
    )
    n_db = connection.execute(text("SELECT COUNT(*) FROM bank_campaign")).scalar_one()
    if n_db != len(df):
        raise ValueError(f"Jumlah baris di PostgreSQL ({n_db}) tidak sama dengan CSV ({len(df)}).")
print("Baris di PostgreSQL:", n_db)

In [ ]:
# Jalankan setiap query terpisah dan hentikan notebook jika ada query yang gagal.
sql_script = (SQL_DIR / "campaign_analysis.sql").read_text(encoding="utf-8")
import re
raw_blocks = re.split(r"(?=-- Q\d+\.)", sql_script)
query_blocks = [block.strip() for block in raw_blocks if re.search(r"-- Q\d+\.", block)]

with engine.connect() as connection:
    for block in query_blocks:
        lines = block.splitlines()
        title = next(line.strip() for line in lines if line.strip().startswith("-- Q"))
        sql_text = "\n".join(
            line for line in lines if line.strip() and not line.lstrip().startswith("--")
        ).strip().rstrip(";")
        if not sql_text:
            continue
        print(f"\n{'=' * 60}\n{title}\n{'=' * 60}")
        display(pd.read_sql(text(sql_text), connection))

## Interpretasi Hasil Query SQL

Query Q0–Q8 di `sql/campaign_analysis.sql` menjawab 8 dari key business questions PRD:

| Query | Business Question | Insight Kunci |
|---|---|---|
| Q0 | Validasi jumlah baris | 45.211 baris = sama dengan CSV sumber |
| Q1 | Tingkat keberhasilan kampanye | Response rate 11,70% (5.289 yes dari 45.211) |
| Q2 | Response rate per pekerjaan | `student` 28,68% dan `retired` 22,79% tertinggi |
| Q3 | Response rate per pendidikan | Tertiary 14,29% > secondary 10,77% > primary 8,53% |
| Q4 | Response rate per kelompok usia | Kelompok <25 dan 65+ cenderung lebih responsif |
| Q5 | Response rate per jenis kontak | `cellular` 14,92% vs `unknown` 4,07% |
| Q6 | Jumlah kontak vs response rate | 1 kontak punya rate tertinggi; makin banyak kontak rate turun |
| Q7 | Kampanye sebelumnya (poutcome) | `success` 64,73%; `failure` 14,29%; `unknown` 4,05% |
| Q8 | Response rate per bulan | `mar` 51,99% (n=477); `may` 6,72% (n=13.766) – perhatikan volume |

> **Catatan metodologis:** Seluruh angka bersifat deskriptif dan **bukan kausal**.
> Perbedaan response rate antar segmen bisa dipengaruhi oleh variabel lain yang tidak dikontrol.
> Validasi dengan eksperimen (A/B test) sebelum mengubah strategi kampanye.

### Query Data Quality
File `sql/data_quality.sql` menyediakan 10 query validasi (DQ-01 s.d. DQ-10) untuk:
- Verifikasi jumlah baris dan missing values
- Pengecekan duplikat, rentang nilai age/campaign/balance
- Distribusi nilai `pdays` dan proporsi `unknown` per kolom kategorikal
